In [ ]:
import sys
import os
import random
import json
import numpy as np
import pandas as pd
import time
import openai
import requests
import re
from tqdm import tqdm
import math
from llm_knowledge_evaluation import discretize_contribution, logic_evaluation, logic_batch_summarize_generate, new_logic_upgrade
from refilter_knowledge import fetch_knowledge, gpt_refilter_knowledge, refilter_knowledge_procedure

# Initial Knowledge Filtering by Reasoning Agent

In [ ]:
# ------------------------exchange dataset：---------------------------#
# #"NZD/USD"#"SGD/USD"#"JPY/USD"#"CNY/USD"#"CHF/USD"#"CAD/USD"#"GBP/USD"#"AUD/USD"
# f''' Positive factors that may lead to an increase in {variable}:
# 1. Factors indicating stronger New Zealand economic fundamentals (e.g., economic growth, trade surplus, capital inflows).
# 2. Policies or events that increase interest rate differentials in favor of New Zealand.
# 3. Government or central bank actions that enhance market confidence or financial stability.
# 4. International cooperation or financial support that reduces economic or financial risk.

# Negative factors that may lead to a decrease in {variable}:
# 1. Factors indicating economic slowdown, recession risk, or financial stress.
# 2. Policies or events that reduce interest rate advantages or increase monetary easing expectations.
# 3. Financial crises, bailouts, or emergency interventions signaling systemic risk.
# 4. Political or economic uncertainty that increases risk aversion.

# Output only ONE single paragraph that directly states the filtering logic for selecting relevant news event descriptions and their positive or negative influence on the target variable. No titles, lists, analysis, or extra text.
# '''

# ------------------------weather dataset：---------------------------#
# variable = "max. PAR" # "PAR" "SWDR"
# knowledge_text = """
# SWDR, PAR, and max. PAR are weather variables strongly influenced by daylight conditions.
# They increase after sunrise, peak around midday (approximately 12:00–14:00 local time),
# and decline to near zero after sunset.
# Cloudy or rainy conditions significantly reduce their intensity compared to clear-sky days.
# """
# prompt_2_details = f'''Positive factors that may lead to an increase in {variable}:
# 1. Clear-sky or mostly clear atmospheric conditions with low cloud coverage.
# 2. Reduced aerosol loading, haze, or dust, allowing more solar radiation to reach the surface.
# 3. Dry and stable atmospheric conditions with relatively low humidity.
# 4. Seasonal or diurnal conditions associated with higher solar elevation angles and stronger incoming solar radiation.

# Negative factors that may lead to a decrease in {variable}:
# 1. Increased cloud coverage, overcast skies, or persistent cloud layers.
# 2. High aerosol concentrations, haze, dust storms, or air pollution events.
# 3. Precipitation, fog, or high humidity conditions that enhance scattering and absorption of solar radiation.
# 4. Large-scale weather systems or atmospheric disturbances that suppress surface solar radiation.

# Output only ONE single paragraph that directly states the filtering logic for selecting relevant news event descriptions and their positive or negative influence on the target variable. No titles, lists, analysis, or extra text.
# '''
# --------------------------------ecl dataset：-------------------------------------#
# variable = "electricity_consumption"
# knowledge_text = '''Calendar-related factors, including day-of-week and public holidays, are known to exert systematic and recurrent effects on electricity demand, 
# with working days generally associated with higher consumption driven by industrial and commercial activity, 
# and weekends or holidays often corresponding to lower demand or distinct load patterns.'''
# prompt_2_details = f'''Positive factors that may lead to an increase in {variable}:
# 1. Large-scale industrial or commercial electricity demand growth, such as new or expanded power supply contracts (GWh-level) for factories, ports, refineries, or manufacturing complexes.
# 2. Economic or industrial activity recovery, including factory restarts, infrastructure operations, or increased logistics and port throughput.
# 3. Policy or market changes that reduce electricity costs for consumers, such as tax reductions, tariff cuts, or enhanced price transparency that may stimulate demand.
# 4. Cross-border electricity trade or grid interconnection developments that increase regional electricity consumption or enable higher local demand absorption.

# Negative factors that may lead to a decrease in {variable}:
# 1. Energy efficiency improvements or demand-reduction measures, such as LED retrofits, efficiency programs, or digital energy management systems that structurally lower electricity consumption.
# 2. Industrial slowdowns, plant shutdowns, or contract terminations that reduce large consumers’ electricity usage.
# 3. Periods of exceptionally high renewable generation penetration that suppress demand for conventional consumption or trigger demand-side curtailment.
# 4. Regulatory or policy-driven demand suppression, including power rationing, mandatory efficiency targets, or structural shifts toward lower electricity intensity.
# Output only ONE single paragraph that directly states the filtering logic for selecting relevant news event descriptions and their positive or negative influence on the target variable. No titles, lists, analysis, or extra text.
# '''
# --------------------------------wind dataset：-------------------------------------#
# prompt_1 = f'''Based on the expert knowledge below, generate logic for filtering exogenous variable information that may influence the wind power data, 
# including Active power, Converter 1 machine-side currents Ia/Ib/Ic, Proximity switch generator speed, Generator speed, Rotor speed, Converter 1 measured generator torque, Stator line voltages Uuv/Uvw/Uwu.'''



In [ ]:
variable = "AUD/USD"
KB_path = "CKFF/Knowledge_text/Standard_Knowledge_base/exchange_knowledge_base.json"
with open(KB_path, "r", encoding="utf-8") as f:
    knowledge_data = json.load(f)
expert_knowledge_entries = [
    entry for entry in knowledge_data if entry.get("knowledge_type") == "variable_description"
]
expert_knowledge_texts = [entry.get("content", "") for entry in expert_knowledge_entries]
knowledge_text = "\n".join(expert_knowledge_texts)

prompt_1 = f'''Based on the expert knowledge below, generate logic for filtering exogenous variable information that may influence the target variable "{variable}".'''

prompt_1_details = f''' Expert Knowledge:
{knowledge_text}

Output only ONE single paragraph that directly states the filtering logic for selecting relevant exogenous variables and their positive or negative influence on the target variable. No titles, lists, analysis, or extra text.
'''

# 基于下面总结的影响因素，生成用于过滤会影响目标变量"{variable}"的新闻事件描述的逻辑
# 会导致{variable}上升的正面因素：
# 会导致{variable}下降的负面因素：
# Australian  Australia UK the UK Canadian Canada Swiss Switzerland Chinese China Japanese Japan Singaporean Singapore
prompt_2 = f'''Based on the summarized influencing factors below, generate logic for filtering news event descriptions that may affect the target variable "{variable}".'''
prompt_2_details = f''' Positive factors that may lead to an increase in {variable}:
1. Factors indicating stronger Australian economic fundamentals (e.g., economic growth, trade surplus, capital inflows).
2. Policies or events that increase interest rate differentials in favor of Australia.
3. Government or central bank actions that enhance market confidence or financial stability.
4. International cooperation or financial support that reduces economic or financial risk.

Negative factors that may lead to a decrease in {variable}:
1. Factors indicating economic slowdown, recession risk, or financial stress.
2. Policies or events that reduce interest rate advantages or increase monetary easing expectations.
3. Financial crises, bailouts, or emergency interventions signaling systemic risk.
4. Political or economic uncertainty that increases risk aversion.

Output only ONE single paragraph that directly states the filtering logic for selecting relevant news event descriptions and their positive or negative influence on the target variable. No titles, lists, analysis, or extra text.
'''

DEEPSEEK_API_KEY = ""
DEEPSEEK_API_URL = ""
def call_deepseek(prompt):
    headers = {
        "Authorization": f"Bearer {DEEPSEEK_API_KEY}",
        "Content-Type": "application/json"
    }

    data = {
        "model": "deepseek-reasoner",
        "messages": [
            {"role": "system", "content": "You are a knowledge filtering logic generation agent."},
            {"role": "user", "content": prompt}
        ],
        "temperature": 0.3
    }

    resp = requests.post(DEEPSEEK_API_URL, headers=headers, json=data)
    resp.raise_for_status()

    return resp.json()["choices"][0]["message"]["content"].strip()

logic_1 = call_deepseek(prompt_1 + prompt_1_details)
logic_2 = call_deepseek(prompt_2 + prompt_2_details)
logic_itr0 = logic_1 + "\n\n" + logic_2



variable_safe = re.sub(r'[^a-zA-Z0-9_-]', '_', variable)
logic_path = f'CKFF/Data_all/logic/exchange/{variable_safe}_itr0.txt'
with open(logic_path, "w", encoding="utf-8") as f:
    f.write(logic_itr0)


In [ ]:
variable =  "AUD/USD"
variable_safe = re.sub(r'[^a-zA-Z0-9_-]', '_', variable)
logic_itr0_path = f'CKFF/Data_all/logic/exchange/{variable_safe}_itr0.txt'
with open(logic_itr0_path, 'r', encoding='utf-8') as f:
    logic_itr0 = f.read()

csv_file_path = f'CKFF/Data_all/Filtered_knowledge/exchange/{variable_safe}_itr0.csv'
header = not os.path.exists(csv_file_path)

KB_path = "CKFF/Knowledge_text/Standard_Knowledge_base/exchange_knowledge_base.json"
        
with open(KB_path, 'r', encoding='utf-8') as file:
    data = json.load(file)
KB_df = pd.DataFrame(data)
KB_df['start_time'] = pd.to_datetime(KB_df['start_time'])
KB_df['end_time'] = pd.to_datetime(KB_df['end_time'])

dates_range= pd.date_range(start=f"1990-01-01", end=f"2010-10-10")
for date in tqdm(dates_range):
    formatted_date = date.strftime('%Y-%m-%d')
    filtered_knoleg_one = fetch_knowledge(date, 3, KB_df)
    if filtered_knoleg_one == "No knowledge found for the prediction date.":
        response = "[]"
    else:
        prompt = f'''Filter relevant knowledge for predicting "{variable}" on {formatted_date}.
    
Filtering logic:
{logic_itr0}

Candidate knowledge entries (each line is one entry; parentheses indicate knowledge_type):
{filtered_knoleg_one}

Task:
Filter the knowledge entries according to the filtering logic for predicting {variable} on {formatted_date}.
Return ONLY the knowledge entries that should be kept.
Do NOT include any knowledge that is excluded, rejected, or disallowed by the filtering logic.

For each kept entry, output a JSON object with exactly three fields:
- content: the original knowledge text
- knowledge_type: the type shown in parentheses for this content
- rationale: a brief explanation of why this knowledge is retained and relevant for predicting the target variable (maximum 30 words)

Output requirements:
- Output strictly as a JSON array.
- Do NOT include explanations for excluded knowledge.
- If no knowledge entries satisfy the filtering logic, output an empty JSON array [].
- Do not include any additional text.
'''

        response = gpt_refilter_knowledge(prompt)
        response = response[response.find("["):response.rfind("]") + 1].replace("\n", "")
    df_one = pd.DataFrame([{"pred_date": formatted_date,"knowledge": response}])
    df_one.to_csv(csv_file_path, mode='a', header=header, index=False, encoding='utf-8')
    header = False

# Train & Test the Model

In [ ]:
# LLM-based Forecasting

# News Logic Upgrade

In [ ]:
variable =  "AUD/USD"
variable_safe = re.sub(r'[^a-zA-Z0-9_-]', '_', variable)
original_logic_path = 'CKFF/Data_all/logic/exchange/AUD_USD_itr0.txt'
with open(original_logic_path, 'r', encoding='utf-8') as f:
    original_logic = f.read()

pred_output_mse_path = 'Forecasting/Val_mse/exchange/knowledge_marginal_contribution_itr0.csv'
df = pd.read_csv(pred_output_mse_path)

csv_file_path_2 = f'CKFF/Data_all/new_constraints/exchange/{variable_safe}_itr0.csv'
header_2 = not os.path.exists(csv_file_path_2)
for i, row in tqdm(df.iterrows(), total=len(df)):
    pred_date = row["pred_date"]
    try:
        knowledge_list = json.loads(row["knowledge_contribution"])
    except json.JSONDecodeError:
        continue

    if len(knowledge_list) == 0:
        continue

    positive_prompt = ""
    negative_prompt = ""

    for k in knowledge_list:
        content = k.get("content", "")
        k_type = k.get("knowledge_type", "")
        delta_str = k.get("delta_mse", "0.000")

        try:
            delta = float(delta_str)
        except ValueError:
            delta = 0.0

        tag = discretize_contribution(delta)
        line = (
            f'- knowledge: "{content}"; type: {k_type}; contribution: {tag}\n'
        )

        if delta < -0.01:
            negative_prompt += line            
        else:
            positive_prompt += line

    logic_tighten_prompt = f'''Refine the knowledge filtering logic for time series forecasting of "{variable}" using empirical feedback.

Forecast date: {pred_date}

Original filtering logic:
{original_logic}

Selected knowledge under the original logic:

Effective:
{positive_prompt}

Ineffective:
{negative_prompt}

Task:
Identify ONLY new or tightened filtering constraints implied by the selection outcomes.
Focus on excluding unnecessary knowledge categories, sources, or time scopes.


IMPORTANT CONSTRAINTS:
- Contribution signals may be used ONLY for internal reasoning, not for output.
- Final constraints MUST NOT reference contributions or selection signals.
- Constraints must be grounded only in observable knowledge attributes (variable, source, temporal scope).

Output requirements:
- Output ONLY the newly added or tightened constraints as a single coherent paragraph.
- Use concise, rule-like language (e.g., "Exclude …", "Ignore …", "Only use …").
- Maximum 30 words total.
- Do NOT restate existing logic.
- No explanations, examples, or metrics.
- Output empty if no new constraint is identified.
'''
    logic_upgrade_response_1 = logic_evaluation(logic_tighten_prompt)
    if logic_upgrade_response_1 == None:
        continue
    df_extended = pd.DataFrame([{'pred_date': pred_date, 'new_constraints': logic_upgrade_response_1}])
    df_extended.to_csv(csv_file_path_2, mode='a', header=header_2, index=False, encoding='utf-8')
    header_2 = False

In [ ]:
variable = "AUD/USD"
variable_safe = re.sub(r'[^a-zA-Z0-9_-]', '_', variable)
original_logic_path = f'CKFF/Data_all/logic/exchange/{variable_safe}_itr1.txt'
with open(original_logic_path, 'r', encoding='utf-8') as f:
    original_logic = f.read()

csv_file_path_2 = f'CKFF/Data_all/new_constraints/exchange/{variable_safe}_itr1.csv'
knowledge_constraints = pd.read_csv(csv_file_path_2)
constraints_list = knowledge_constraints["new_constraints"].tolist()

summaries_output_path = f"CKFF/Data_all/new_constraints/exchange/{variable_safe}_itr1_summarized_constraints.txt"

batch_size = 30
num_batches = math.ceil(len(constraints_list) / batch_size)

batches = []
for i in range(num_batches):
    batch_constraints = constraints_list[i*batch_size : (i+1)*batch_size]
    batch_text = "".join([f"- {c}\n" for c in batch_constraints])
    batches.append(batch_text)

summaries = []
for batch_text in tqdm(batches):
    logic_batch_summarize_prompt = f'''Summarize the following newly identified knowledge filtering constraints across 30 forecast dates into a single, concise, and general constraint.

Constraints:
{batch_text}

Task:
Summarize all constraints across into one concise, implementable rule.

Guidelines:
- Identify common patterns or recurring rules.
- Merge overlapping or compatible constraints.
- Discard contradictory, overly specific, or date-dependent constraints.

Output requirements:
- Output ONLY one concise sentence capturing the general constraint for this batch.
- Maximum 50 words.
- Write as a rule-like statement (e.g., "Restrict ...", "Exclude ...", "Require ...").
- Do NOT reference dates, individual cases, or empirical results.
- Output empty if no generalizable constraint is identified.
'''
    logic_upgrade_response_2 = logic_batch_summarize_generate(logic_batch_summarize_prompt)
    if logic_upgrade_response_2 == None:
        continue
    summaries.append(logic_upgrade_response_2)
    
summarized_constraints_text = "\n".join([f"- {s}" for s in summaries if s])
with open(summaries_output_path, "w", encoding="utf-8") as f:
    f.write(summarized_constraints_text)


In [ ]:
logic_upgrade_prompt = f'''Generate the updated knowledge filtering logic for the variable "{variable}" by integrating the original logic with the following general constraints from multiple forecast date batches.

Original filtering logic:
{original_logic}

Summarized batch constraints:
{summarized_constraints_text}

Task:
Integrate these constraints into the original logic to produce a single, unified, stable knowledge filtering logic for the variable "{variable}".

Guidelines:
- Prioritize knowledge explicitly about "{variable}", allowing broader context only when its relevance is explicit.
- Preserve the structure and intent of the original logic.
- Incorporate recurring or compatible constraints as general rules.
- Merge overlapping constraints into concise, implementable conditions.
- Discard contradictory, overly specific, or date-dependent constraints.

Output requirements:
- Output ONLY the updated knowledge filtering logic as a single coherent paragraph.
- Write as a coherent, implementable system rule.
- Do NOT reference dates, individual cases, or empirical evaluation.
- Do NOT explain or justify changes.
'''
logic_upgrade_response_3 = new_logic_upgrade(logic_upgrade_prompt)
logic_path = f'CKFF/Data_all/logic/exchange/{variable_safe}_itr2.txt'
with open(logic_path, "w", encoding="utf-8") as f:
    f.write(logic_upgrade_response_3)


# Re-Filter Knowledge in the Next Iteration

In [ ]:
variable =  "AUD/USD"
variable_safe = re.sub(r'[^a-zA-Z0-9_-]', '_', variable)
itr_num = 1
new_logic_path = f'CKFF/Data_all/logic/exchange/{variable_safe}_itr{itr_num}.txt'
with open(new_logic_path, 'r', encoding='utf-8') as f:
    new_logic = f.read()
KB_path = "CKFF/Knowledge_text/Standard_Knowledge_base/exchange_knowledge_base.json"
csv_file_path = f'CKFF/Data_all/Filtered_knowledge/exchange/{variable_safe}_itr{itr_num}.csv'

refilter_knowledge_procedure(variable, KB_path, csv_file_path, new_logic)